# `config.ipynb` - Configuracion centralizada del proyecto

Carga y expone como constantes de Python todos los parametros operativos
del pipeline (API, ubicaciones, umbrales de calidad de datos, rutas de
almacenamiento) definidos en `config.yaml`, en la raiz del entregable.

Cubre el requisito de la libreria **pyyaml** especificado en el enunciado
("Configuracion: gestion centralizada de parametros operativos") y evita
"numeros magicos" repartidos por el resto de modulos: cualquier ajuste de
ciudad, umbral o politica de reintentos se hace en un unico fichero YAML,
sin tocar codigo.

> **Decision de diseno**: las claves de la API de Open-Meteo no requieren
> autenticacion (segun enunciado), por lo que no aplica aqui el patron de
> `API_KEY` con variable de entorno + valor de respaldo. Se mantiene, en
> cambio, el patron de `USER_AGENT_API` configurable por variable de
> entorno, por si la API decidiera bloquear el User-Agent por defecto de
> `requests` en el futuro.

> Depende de: `config.yaml` (fichero de configuracion en la raiz del
> entregable). No depende de otros notebooks.

In [1]:
import os
import yaml
from pathlib import Path

## Localizacion y carga del fichero de configuracion

In [2]:
def localizar_config_yaml():
    """Busca config.yaml subiendo desde el directorio de trabajo actual.

    Permite que este notebook funcione tanto si se ejecuta desde la raiz
    del entregable como desde dentro de climate_engine/ (patron de prueba
    independiente descrito en el documento de convenciones).

    Devuelve: ruta absoluta (Path) al fichero config.yaml encontrado.
    """
    directorio = Path.cwd()
    for _ in range(4):
        candidato = directorio / "config.yaml"
        if candidato.exists():
            return candidato
        directorio = directorio.parent
    raise FileNotFoundError("No se encontro config.yaml en los directorios superiores.")


RUTA_CONFIG_YAML = localizar_config_yaml()
with open(RUTA_CONFIG_YAML, "r", encoding="utf-8") as _f:
    CONFIG = yaml.safe_load(_f)

# Directorio raiz del entregable (donde vive config.yaml), usado para anclar
# todas las rutas relativas de datos y logs de forma robusta.
RAIZ_PROYECTO = RUTA_CONFIG_YAML.parent

## Funcion principal

In [3]:
# --- Parametros de la API de Open-Meteo ---
# Patron de doble via (igual que USER_AGENT_API): en operacion normal se usa
# la URL oficial de Open-Meteo. La variable de entorno CLIMATE_ENGINE_API_URL_OVERRIDE
# permite apuntar a un servidor equivalente (staging, sandbox de pruebas sin
# salida a internet) sin tocar el codigo. Ver README, seccion "Notas importantes".
API_URL_BASE = os.environ.get("CLIMATE_ENGINE_API_URL_OVERRIDE", CONFIG["api"]["url_base"])
API_VARIABLES_HORARIAS = CONFIG["api"]["variables_horarias"]
API_FORECAST_DAYS = CONFIG["api"]["forecast_days"]
API_TIMEZONE = CONFIG["api"]["timezone"]
API_TIMEOUT = CONFIG["api"]["timeout_segundos"]
API_MAX_REINTENTOS = CONFIG["api"]["max_reintentos"]
API_BACKOFF_FACTOR = CONFIG["api"]["backoff_factor"]
API_STATUS_FORCELIST = CONFIG["api"]["status_forcelist"]

# Patron de doble via para el User-Agent (variable de entorno con valor de
# respaldo real), igual que el patron de API_KEY del documento de convenciones.
USER_AGENT_API = os.environ.get(
    CONFIG["api"]["user_agent_var_entorno"],
    CONFIG["api"]["user_agent_valor_defecto"],
)

# --- Catalogo de ubicaciones (minimo 5 ciudades peninsulares, por enunciado) ---
UBICACIONES = CONFIG["ubicaciones"]

# --- Umbrales del contrato de calidad de datos (Silver) ---
RANGO_TEMPERATURA_C = tuple(CONFIG["calidad_datos"]["rango_temperatura_c"])
RANGO_HUMEDAD_PCT = tuple(CONFIG["calidad_datos"]["rango_humedad_pct"])
RANGO_PRECIPITACION_MM = tuple(CONFIG["calidad_datos"]["rango_precipitacion_mm"])
RANGO_VIENTO_KMH = tuple(CONFIG["calidad_datos"]["rango_viento_kmh"])
RANGO_PRESION_HPA = tuple(CONFIG["calidad_datos"]["rango_presion_hpa"])

# --- Rutas de almacenamiento por capas (ancladas a la raiz del proyecto) ---
RUTA_BRONZE = RAIZ_PROYECTO / CONFIG["rutas"]["bronze"]
RUTA_SILVER = RAIZ_PROYECTO / CONFIG["rutas"]["silver"]
RUTA_GOLD = RAIZ_PROYECTO / CONFIG["rutas"]["gold"]
RUTA_CUARENTENA = RAIZ_PROYECTO / CONFIG["rutas"]["cuarentena"]
RUTA_LOG = RAIZ_PROYECTO / CONFIG["rutas"]["logs"]

for _ruta in (RUTA_BRONZE, RUTA_SILVER, RUTA_GOLD, RUTA_CUARENTENA, RUTA_LOG.parent):
    _ruta.mkdir(parents=True, exist_ok=True)

# --- Parametros de analitica (Gold) ---
VENTANA_MEDIA_MOVIL_HORAS = CONFIG["analitica"]["ventana_media_movil_horas"]

## Prueba rapida

Se listan los parametros cargados para confirmar visualmente que `config.yaml` se ha leido correctamente y que las rutas de datos existen en disco.

In [4]:
print(f"Fichero de configuracion: {RUTA_CONFIG_YAML}")
print(f"Raiz del proyecto:        {RAIZ_PROYECTO}")
print(f"Ubicaciones cargadas ({len(UBICACIONES)}): {list(UBICACIONES.keys())}")
print(f"Variables horarias:       {API_VARIABLES_HORARIAS}")
print(f"Ventana de prediccion:    {API_FORECAST_DAYS} dias, zona horaria {API_TIMEZONE}")
print(f"Politica de reintentos:   {API_MAX_REINTENTOS} reintentos, backoff x{API_BACKOFF_FACTOR}, codigos {API_STATUS_FORCELIST}")
print(f"Rango temperatura valido: {RANGO_TEMPERATURA_C}")
print(f"Rango humedad valido:     {RANGO_HUMEDAD_PCT}")
print(f"Ruta bronze:  {RUTA_BRONZE}  (existe: {RUTA_BRONZE.exists()})")
print(f"Ruta silver:  {RUTA_SILVER}  (existe: {RUTA_SILVER.exists()})")
print(f"Ruta gold:    {RUTA_GOLD}  (existe: {RUTA_GOLD.exists()})")
print(f"URL base API: {API_URL_BASE}")

assert len(UBICACIONES) >= 5, "Se requieren al menos 5 ubicaciones de referencia."
for _nombre, _coords in UBICACIONES.items():
    assert "lat" in _coords and "lon" in _coords, f"Faltan coordenadas para {_nombre}"
print("Validacion de catalogo de ubicaciones: OK")

Fichero de configuracion: /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/config.yaml
Raiz del proyecto:        /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco
Ubicaciones cargadas (5): ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
Variables horarias:       ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'surface_pressure']
Ventana de prediccion:    14 dias, zona horaria UTC
Politica de reintentos:   3 reintentos, backoff x2, codigos [429, 500, 502, 503, 504]
Rango temperatura valido: (-50.0, 60.0)
Rango humedad valido:     (0.0, 100.0)
Ruta bronze:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/bronze  (existe: True)
Ruta silver:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/silver  (existe: True)
Ruta gold:    /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/gold  (existe: True)
URL base API: https://api.open-me

---
✅ **`config.ipynb` verificado/a.**